<a href="https://colab.research.google.com/github/AlperYildirim1/crt-fourier-transformer-addition/blob/main/crt_fourier_transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Pythia-6.9B Addition: Paper-Ready Persistent Fourier Ablation
#
# Suggested notebook name:
#   pythia_units_digit_CRT_phase_causal_ablation_paper_ready.ipynb
#
# Runtime:
#   A100 recommended. This notebook loads Pythia-6.9B and runs live-model
#   interventions with residual-stream hooks.
#
# Main claims this notebook can support:
#   1. Natural errors preserve base-10 residues much more than exact value.
#   2. Persistent removal of the learned low-period Fourier span T2/T5/T10
#      collapses addition / units-digit accuracy.
#   3. Matched-rank random removals do not collapse performance.
#   4. Period-wise removals show residue-specific structure, especially:
#        T5 removal preserves mod2/parity but damages mod5 and mod10.
#   5. Wrong-frequency controls T3/T7 are confounded by overlap/leakage;
#      orthogonalizing them to an empirical helix+magnitude span greatly
#      reduces their damage.
#
# Important non-claim:
#   This does NOT prove causal phase rotation. It supports:
#     causal Fourier/CRT span + decodable phase-like structure.
# ============================================================

# Optional Colab install cell:
# !pip install -q transformers accelerate tqdm scikit-learn pandas

import os
import json
import random
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "EleutherAI/pythia-6.9b"

# Mode controls eval size and output saving.
#   "smoke": quick check on 1000 baseline-correct examples.
#   "paper": full baseline-correct eval set.
RUN_MODE = "paper"  # change to "smoke" for debugging

BASE_SEED = 42
A_MAX = 99
B_MAX = 99
BATCH_SIZE = 32
RIDGE_ALPHA = 1.0

# Fit subspaces on this many baseline-correct examples.
# Keep fixed for comparability. 4000 was used in pilot experiments.
FIT_N = 4000

if RUN_MODE == "smoke":
    MAX_EVAL_EXAMPLES = 1000
    RANDOM_CONTROL_SEEDS = [0]
else:
    MAX_EVAL_EXAMPLES = None
    # Random controls are the only inherently stochastic part in the main causal test.
    # These seeds give mean/std for matched-rank random baselines.
    RANDOM_CONTROL_SEEDS = [0, 1, 2, 3, 4]

# Persistent ablation collects/fits residuals from these layers onward.
LAYER_MIN = 10

# Main causal start layer for period-wise ablation.
L0 = 17

# Main low-period Fourier span used in headline persistent ablation.
PERIODS_MAIN = [2, 5, 10]
INCLUDE_LINEAR_MAIN = False

# Empirical richer span used for wrong-frequency leakage controls.
REAL_PERIODS = [2, 2.5, 5, 10, 20, 100]

# Whether to save CSVs to local runtime.
SAVE_RESULTS = True
OUT_DIR = f"/content/pythia_crt_ablation_{RUN_MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"


# ============================================================
# UTILS
# ============================================================

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def save_df(df, name):
    if not SAVE_RESULTS:
        return
    os.makedirs(OUT_DIR, exist_ok=True)
    path = os.path.join(OUT_DIR, name)
    df.to_csv(path, index=False)
    print("saved:", path)


set_seed(BASE_SEED)

print_section("ENVIRONMENT")
print("run mode:", RUN_MODE)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("gpu memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("OUT_DIR:", OUT_DIR if SAVE_RESULTS else None)


# ============================================================
# LOAD MODEL
# ============================================================

print_section("LOAD TOKENIZER")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print("pad_token:", repr(tokenizer.pad_token), tokenizer.pad_token_id)
print("padding_side:", tokenizer.padding_side)

print_section("LOAD MODEL")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,  # transformers may warn; works on current Colab
    device_map="auto",
    low_cpu_mem_usage=True,
).eval()

model.config.pad_token_id = tokenizer.pad_token_id
N_LAYERS = model.config.num_hidden_layers

print("model:", MODEL_NAME)
print("num layers:", N_LAYERS)
print("hidden size:", model.config.hidden_size)
print("dtype:", next(model.parameters()).dtype)
print("device:", model.device)


# ============================================================
# DATASET HELPERS
# ============================================================

def answer_token_id(n):
    ids = tokenizer(str(int(n)), add_special_tokens=False)["input_ids"]
    if len(ids) != 1:
        return None
    return int(ids[0])


def make_examples(a_max=A_MAX, b_max=B_MAX):
    rows = []
    bad = []
    for a in range(a_max + 1):
        for b in range(b_max + 1):
            s = a + b
            tid = answer_token_id(s)
            if tid is None:
                bad.append((a, b, s))
                continue
            rows.append({
                "a": int(a),
                "b": int(b),
                "sum": int(s),
                "mod2": int(s % 2),
                "mod3": int(s % 3),
                "mod5": int(s % 5),
                "mod7": int(s % 7),
                "mod10": int(s % 10),
                "prompt": f"Output ONLY a number. {a}+{b}=",
                "target_token_id": int(tid),
            })
    return rows, bad


def get_last_positions(attention_mask):
    return attention_mask.sum(dim=1) - 1


def pred_token_to_int(pred_id):
    txt = tokenizer.decode([int(pred_id)]).strip()
    try:
        return int(txt)
    except Exception:
        return None


examples, bad_answer_examples = make_examples()

print_section("DATASET")
print("examples:", len(examples))
print("bad answer examples:", len(bad_answer_examples))
print("first example:", examples[0])
print("last example:", examples[-1])


# ============================================================
# SANITY CHECK: TOKEN POSITIONS
# ============================================================

print_section("TOKENIZATION SANITY CHECK")

lens = {
    len(tokenizer(f"Output ONLY a number. {a}+{b}=", add_special_tokens=False).input_ids)
    for a in [0, 5, 17, 99]
    for b in [0, 9, 42, 99]
}
print("unique prompt lengths:", lens)

for a in [0, 5, 17, 99]:
    for b in [0, 9, 42, 99]:
        text = f"Output ONLY a number. {a}+{b}="
        ids = tokenizer(text, add_special_tokens=False).input_ids
        toks = [tokenizer.decode([i]) for i in ids]
        print(f"{text!r} | len={len(ids)} | last={toks[-1]!r} | toks={toks}")


# ============================================================
# BASELINE AND NATURAL ERROR ANALYSIS
# ============================================================

@torch.no_grad()
def baseline_predictions(rows):
    out_rows = []
    for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="baseline predictions"):
        batch = rows[start:start + BATCH_SIZE]
        enc = tokenizer(
            [x["prompt"] for x in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)
        lp = get_last_positions(enc["attention_mask"])
        bi = torch.arange(len(batch), device=model.device)
        logits = model(**enc, use_cache=False).logits[bi, lp]
        pred_ids = logits.argmax(-1).detach().cpu().tolist()
        for pid, ex in zip(pred_ids, batch):
            pn = pred_token_to_int(pid)
            out_rows.append({**ex, "pred_token_id": int(pid), "pred": pn, "correct": int(pid) == int(ex["target_token_id"])})
    return out_rows


print_section("BASELINE PREDICTIONS + NATURAL ERRORS")

pred_rows = baseline_predictions(examples)
baseline_correct_examples = [r for r in pred_rows if r["correct"]]
wrong_rows = [r for r in pred_rows if not r["correct"]]
parse_wrong = [r for r in wrong_rows if r["pred"] is not None]

print("baseline correct:", len(baseline_correct_examples), "/", len(examples))
print("baseline exact acc:", len(baseline_correct_examples) / len(examples))
print("wrong total:", len(wrong_rows))
print("parseable wrong:", len(parse_wrong))
print("unparseable wrong:", len(wrong_rows) - len(parse_wrong))

if MAX_EVAL_EXAMPLES is None:
    eval_examples = baseline_correct_examples
else:
    eval_examples = baseline_correct_examples[:MAX_EVAL_EXAMPLES]
print("eval examples:", len(eval_examples))

# Natural error analysis.
if len(parse_wrong):
    diffs = np.array([r["pred"] - r["sum"] for r in parse_wrong], dtype=int)
    print("\nTop natural error shifts: pred - true")
    v, c = np.unique(diffs, return_counts=True)
    for i in np.argsort(-c)[:20]:
        print(f"  {int(v[i]):+d}: {int(c[i])} ({c[i] / len(diffs) * 100:.1f}%)")

    print("\nResidue preserved among parseable wrong answers:")
    for k in [2, 3, 5, 7, 10]:
        keep = np.mean([r["pred"] % k == r["sum"] % k for r in parse_wrong])
        print(f"  mod{k:<2} still correct: {keep * 100:.1f}%")

    ad = np.abs(diffs)
    print("\nError type:")
    print(f"  multiple of 10 error     (units digit correct): {np.mean(ad % 10 == 0) * 100:.1f}%")
    print(f"  small non-multiple <10   (units digit wrong):   {np.mean((ad % 10 != 0) & (ad < 10)) * 100:.1f}%")
    print(f"  large non-multiple >=10  (mixed error):         {np.mean((ad % 10 != 0) & (ad >= 10)) * 100:.1f}%")

# Save baseline predictions.
save_df(pd.DataFrame(pred_rows), "baseline_predictions.csv")


# ============================================================
# FOURIER BASIS / SUBSPACE HELPERS
# ============================================================

def make_fourier_basis(sums, periods, include_linear=False):
    s = np.asarray(sums, dtype=np.float64)
    cols = []
    if include_linear:
        z = (s - s.mean()) / (s.std() + 1e-12)
        cols.append(z)
    for T in periods:
        theta = 2.0 * np.pi * s / float(T)
        cols.append(np.cos(theta))
        cols.append(np.sin(theta))
    return np.column_stack(cols)


def fit_Q(X, sums, periods=PERIODS_MAIN, include_linear=INCLUDE_LINEAR_MAIN):
    Y = make_fourier_basis(sums, periods=periods, include_linear=include_linear)
    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(Y, X)
    W = reg.coef_  # [D, K]
    Q, _ = np.linalg.qr(W)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)


def fit_lin_Q(X, sums):
    s = np.asarray(sums, dtype=np.float64)
    z = ((s - s.mean()) / (s.std() + 1e-12)).reshape(-1, 1)
    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True)
    reg.fit(z, X)
    W = reg.coef_
    Q, _ = np.linalg.qr(W)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)


def rand_Q(D, k, seed=0):
    g = torch.Generator(device="cpu").manual_seed(seed)
    A = torch.randn(D, k, generator=g, dtype=torch.float32)
    Q, _ = torch.linalg.qr(A)
    return Q.to(dtype=torch.float16, device=model.device)


def orthobasis(Qs):
    M = torch.cat(Qs, dim=1).float()
    Q, _ = torch.linalg.qr(M)
    return Q.to(dtype=torch.float16, device=model.device)


def princ_cos(A, B):
    return torch.linalg.svdvals(A.float().T @ B.float())


def orth_to(Qw, H):
    R = Qw.float() - H.float() @ (H.float().T @ Qw.float())
    norms = torch.linalg.norm(R, dim=0)
    Q, _ = torch.linalg.qr(R)
    return Q[:, :Qw.shape[1]].to(dtype=torch.float16, device=model.device), norms.detach().cpu().numpy()


# ============================================================
# COLLECT RESIDUALS FOR FITTING Q_l
# ============================================================

def collect_resids(rows, layers):
    store = {l: [] for l in layers}
    sums = []
    buf = {}

    def make_hook(L):
        def hook(module, inputs, output):
            hs = output[0] if isinstance(output, tuple) else output
            buf[L] = hs.detach()
        return hook

    handles = [model.gpt_neox.layers[l].register_forward_hook(make_hook(l)) for l in layers]
    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc="collect residuals"):
            batch = rows[start:start + BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)
            lp = get_last_positions(enc["attention_mask"])
            bi = torch.arange(len(batch), device=model.device)
            buf.clear()
            with torch.no_grad():
                model(**enc, use_cache=False)
            for l in layers:
                if l not in buf:
                    raise RuntimeError(f"Missing hook output for layer {l}")
                store[l].append(buf[l][bi, lp].float().cpu().numpy())
            sums.extend([x["sum"] for x in batch])
    finally:
        for h in handles:
            h.remove()
    return {l: np.concatenate(store[l], axis=0) for l in layers}, np.array(sums, dtype=np.float64)


# ============================================================
# PERSISTENT RESIDUAL ABLATION
# ============================================================

_lp_g = None
_bi_g = None


def make_resid_hook(Q):
    def hook(module, inputs, output):
        global _lp_g, _bi_g
        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None
        v = hs[_bi_g, _lp_g]
        hs[_bi_g, _lp_g] = v - (v @ Q) @ Q.T
        return hs if rest is None else (hs,) + rest
    return hook


@torch.no_grad()
def eval_persistent(rows, Q_by_layer, label="", ks=(2, 3, 5, 7, 10), show_progress=True):
    global _lp_g, _bi_g
    handles = [model.gpt_neox.layers[l].register_forward_hook(make_resid_hook(Q)) for l, Q in Q_by_layer.items()]

    n = exact = parseable = 0
    hit = {k: 0 for k in ks}
    diffs_all = []
    diffs_wrong = []

    iterator = range(0, len(rows), BATCH_SIZE)
    if show_progress:
        iterator = tqdm(iterator, desc=f"eval persistent {label}")

    try:
        for start in iterator:
            batch = rows[start:start + BATCH_SIZE]
            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)
            _lp_g = get_last_positions(enc["attention_mask"])
            _bi_g = torch.arange(len(batch), device=model.device)
            logits = model(**enc, use_cache=False).logits[_bi_g, _lp_g]
            pred_ids = logits.argmax(-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                n += 1
                is_exact = int(pred_id) == int(ex["target_token_id"])
                exact += int(is_exact)
                pred_num = pred_token_to_int(pred_id)
                if pred_num is None:
                    continue
                parseable += 1
                for k in ks:
                    hit[k] += int(pred_num % k == ex["sum"] % k)
                diff = pred_num - ex["sum"]
                diffs_all.append(diff)
                if not is_exact:
                    diffs_wrong.append(diff)
    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)
    diffs_all = np.array(diffs_all)
    diffs_wrong = np.array(diffs_wrong)

    out = {
        "n": n,
        "exact": exact / n_safe,
        "parseable": parseable / n_safe,
        "median_abs_err_all": float(np.median(np.abs(diffs_all))) if len(diffs_all) else None,
        "median_abs_err_wrong": float(np.median(np.abs(diffs_wrong))) if len(diffs_wrong) else None,
        "frac_wrong_units": float(np.mean(np.abs(diffs_all) % 10 != 0)) if len(diffs_all) else None,
    }
    for k in ks:
        out[f"mod{k}"] = hit[k] / n_safe
    return out


# ============================================================
# FIT RESIDUAL SUBSPACES
# ============================================================

print_section("COLLECT RESIDUALS AND FIT MAIN Qs")

layers = list(range(LAYER_MIN, N_LAYERS))
print("layers:", layers)
print("fit examples:", min(FIT_N, len(baseline_correct_examples)))

Xres, Sres = collect_resids(baseline_correct_examples[:FIT_N], layers)

print("Residual shapes:")
for l in [layers[0], L0, layers[-1]]:
    print(f"layer {l}: X={Xres[l].shape}")

print("Fitting main helix Q per layer...")
Qh = {l: fit_Q(Xres[l], Sres, periods=PERIODS_MAIN, include_linear=INCLUDE_LINEAR_MAIN) for l in layers}

print("Fitting matched-rank random Q per layer for sweep seed 0...")
Qr_seed0 = {l: rand_Q(Xres[l].shape[1], Qh[l].shape[1], seed=0 * 1000 + l) for l in layers}


# ============================================================
# TEST 1: PERSISTENT ABLATION SWEEP
# ============================================================

print_section("TEST 1: PERSISTENT RESIDUAL ABLATION SWEEP")

sweep_rows = []
print(f"{'L_start':>7} {'helix_exact':>12} {'rand_exact':>11} {'helix_m10':>10} {'rand_m10':>9} {'n_layers':>8}")
print("-" * 80)

for L_start in range(N_LAYERS - 1, LAYER_MIN - 1, -3):
    sub_h = {l: Qh[l] for l in layers if l >= L_start}
    sub_r = {l: Qr_seed0[l] for l in layers if l >= L_start}
    rh = eval_persistent(eval_examples, sub_h, label=f"helix L>={L_start}")
    rr = eval_persistent(eval_examples, sub_r, label=f"random L>={L_start}")

    print(f"{L_start:>7} {rh['exact']:>12.4f} {rr['exact']:>11.4f} {rh['mod10']:>10.4f} {rr['mod10']:>9.4f} {len(sub_h):>8}")

    sweep_rows.append({"condition": "helix", "L_start": L_start, "n_layers": len(sub_h), **rh})
    sweep_rows.append({"condition": "random_seed0", "L_start": L_start, "n_layers": len(sub_r), **rr})

sweep_df = pd.DataFrame(sweep_rows)
save_df(sweep_df, "test1_persistent_sweep.csv")


# ============================================================
# TEST 2: PERIOD-WISE PERSISTENT ABLATION
# ============================================================

print_section("TEST 2: PERIOD-WISE PERSISTENT ABLATION")

SETS = [
    ("T2", [2]),
    ("T5", [5]),
    ("T10", [10]),
    ("T3_ctrl", [3]),
    ("T7_ctrl", [7]),
    ("T2T5T10", [2, 5, 10]),
]

periodwise_rows = []
print(f"{'subspace':10} {'exact':>7} {'mod2':>6} {'mod3':>6} {'mod5':>6} {'mod7':>6} {'mod10':>6} {'med|err|wrong':>14} {'wrong_units':>12} {'parse':>7}")
print("-" * 110)

for name, periods in SETS:
    Q = {l: fit_Q(Xres[l], Sres, periods=periods, include_linear=False) for l in layers if l >= L0}
    r = eval_persistent(eval_examples, Q, label=name)
    periodwise_rows.append({"subspace": name, "periods": str(periods), "random_seed": None, **r})
    print(f"{name:10} {r['exact']:>7.3f} {r['mod2']:>6.3f} {r['mod3']:>6.3f} {r['mod5']:>6.3f} {r['mod7']:>6.3f} {r['mod10']:>6.3f} {(r['median_abs_err_wrong'] or 0):>14.1f} {(r['frac_wrong_units'] or 0):>12.3f} {r['parseable']:>7.3f}")

# Random rank-6 controls across seeds.
for seed in RANDOM_CONTROL_SEEDS:
    Q = {l: rand_Q(Xres[l].shape[1], 6, seed=seed * 1000 + l) for l in layers if l >= L0}
    r = eval_persistent(eval_examples, Q, label=f"RANDOM_rank6_seed{seed}")
    periodwise_rows.append({"subspace": "RANDOM_rank6", "periods": None, "random_seed": seed, **r})
    print(f"{'RANDOM'+str(seed):10} {r['exact']:>7.3f} {r['mod2']:>6.3f} {r['mod3']:>6.3f} {r['mod5']:>6.3f} {r['mod7']:>6.3f} {r['mod10']:>6.3f} {(r['median_abs_err_wrong'] or 0):>14.1f} {(r['frac_wrong_units'] or 0):>12.3f} {r['parseable']:>7.3f}")

periodwise_df = pd.DataFrame(periodwise_rows)
save_df(periodwise_df, "test2_periodwise_ablation.csv")

# Random summary.
rand6 = periodwise_df[periodwise_df["subspace"] == "RANDOM_rank6"]
if len(rand6):
    print("\nRANDOM rank-6 summary:")
    print(rand6[["exact", "mod2", "mod3", "mod5", "mod7", "mod10"]].agg(["mean", "std"]))


# ============================================================
# TEST 3: WRONG-FREQUENCY ORTHOGONALIZED CONTROL
# ============================================================

print_section("TEST 3: WRONG-FREQUENCY OVERLAP / ORTHOGONALIZED CONTROL")

# Helix span: linear + T2/T5/T10/T100.
Hspan = {}
for l in layers:
    if l < L0:
        continue
    Hspan[l] = orthobasis(
        [fit_Q(Xres[l], Sres, periods=[T], include_linear=False) for T in [2, 5, 10, 100]]
        + [fit_lin_Q(Xres[l], Sres)]
    )

# Rank-2 random controls across seeds.
rank2_rows = []
for seed in RANDOM_CONTROL_SEEDS:
    Qr2 = {l: rand_Q(Xres[l].shape[1], 2, seed=10_000 + seed * 1000 + l) for l in Hspan}
    r = eval_persistent(eval_examples, Qr2, label=f"random rank2 seed{seed}")
    rank2_rows.append({"condition": "RANDOM_rank2", "random_seed": seed, **r})
rank2_df = pd.DataFrame(rank2_rows)
save_df(rank2_df, "test3_random_rank2_controls.csv")
print("RANDOM rank-2 summary:")
print(rank2_df[["exact", "mod2", "mod3", "mod5", "mod7", "mod10"]].agg(["mean", "std"]))

wrong_rows = []
print(f"\n{'freq':<6} {'cos@L0':<18} {'resid_norm@L0':<24} {'raw_exact':>10} {'orth_exact':>11} {'raw_m10':>9} {'orth_m10':>10}")
print("-" * 100)

for T in [3, 7]:
    Qw = {l: fit_Q(Xres[l], Sres, periods=[T], include_linear=False) for l in Hspan}
    Qwort = {}
    norms_by_layer = {}
    for l in Hspan:
        Q_orth, norms = orth_to(Qw[l], Hspan[l])
        Qwort[l] = Q_orth
        norms_by_layer[l] = norms

    cos_l = princ_cos(Qw[L0], Hspan[L0]).detach().cpu().numpy()
    norms_l = norms_by_layer[L0]
    r_raw = eval_persistent(eval_examples, Qw, label=f"T{T} raw")
    r_orth = eval_persistent(eval_examples, Qwort, label=f"T{T} orth")

    print(f"T{T:<5} {[round(float(c), 3) for c in cos_l]!s:<18} {[round(float(n), 3) for n in norms_l]!s:<24} {r_raw['exact']:>10.3f} {r_orth['exact']:>11.3f} {r_raw['mod10']:>9.3f} {r_orth['mod10']:>10.3f}")

    wrong_rows.append({"freq": T, "condition": "raw", "cos_L0": str([float(c) for c in cos_l]), "resid_norm_L0": str([float(n) for n in norms_l]), **r_raw})
    wrong_rows.append({"freq": T, "condition": "orth_to_Hspan", "cos_L0": str([float(c) for c in cos_l]), "resid_norm_L0": str([float(n) for n in norms_l]), **r_orth})

wrong_df = pd.DataFrame(wrong_rows)
save_df(wrong_df, "test3_wrong_frequency_orthogonalized.csv")


# ============================================================
# TEST 4: EMPIRICAL SPECTRUM + RICHER Hfull CONTROL
# ============================================================

print_section("TEST 4: EMPIRICAL SPECTRUM + RICHER Hfull CONTROL")


def sum_spectrum(X, sums, smax=198):
    sums = np.asarray(sums, dtype=int)
    D = X.shape[1]
    M = np.zeros((smax + 1, D), dtype=np.float64)
    c = np.zeros(smax + 1, dtype=np.float64)
    np.add.at(M, sums, X)
    np.add.at(c, sums, 1)
    keep = c > 0
    M = M[keep] - X.mean(0, keepdims=True)
    F = np.abs(np.fft.rfft(M, axis=0))
    mag = np.linalg.norm(F, axis=1)
    freq = np.fft.rfftfreq(keep.sum())
    per = 1.0 / np.maximum(freq, 1e-9)
    top = np.argsort(mag)[::-1][1:12]
    return [(round(float(per[i]), 2), round(float(mag[i]), 1)) for i in top]


spectrum = sum_spectrum(Xres[L0], Sres)
print("empirical outlier periods @L0:", spectrum)
save_df(pd.DataFrame(spectrum, columns=["period", "magnitude"]), "test4_empirical_spectrum_L0.csv")


def mag_basis(X, sums, k=4):
    Xc = X - X.mean(0, keepdims=True)
    pca = PCA(n_components=30, random_state=0)
    Z = pca.fit_transform(Xc)
    corr = np.array([abs(np.corrcoef(Z[:, j], sums)[0, 1]) for j in range(30)])
    top = np.argsort(corr)[::-1][:k]
    comps = pca.components_[top]
    Q, _ = np.linalg.qr(comps.T)
    return torch.tensor(Q, dtype=torch.float16, device=model.device)


Hfull = {}
for l in Hspan:
    Hfull[l] = orthobasis(
        [fit_Q(Xres[l], Sres, periods=[T], include_linear=False) for T in REAL_PERIODS]
        + [fit_lin_Q(Xres[l], Sres), mag_basis(Xres[l], Sres)]
    )

hfull_rows = []
for T in [3, 7]:
    Qw = {l: fit_Q(Xres[l], Sres, periods=[T], include_linear=False) for l in Hfull}
    Qwo = {l: orth_to(Qw[l], Hfull[l])[0] for l in Hfull}
    r_raw = eval_persistent(eval_examples, Qw, label=f"T{T} raw Hfull")
    r_orth = eval_persistent(eval_examples, Qwo, label=f"T{T} orth Hfull")
    print(f"T{T}: raw_exact={r_raw['exact']:.3f} raw_m10={r_raw['mod10']:.3f} | orth_to_FULL_exact={r_orth['exact']:.3f} orth_to_FULL_m10={r_orth['mod10']:.3f}")
    hfull_rows.append({"freq": T, "condition": "raw", **r_raw})
    hfull_rows.append({"freq": T, "condition": "orth_to_FULL", **r_orth})

hfull_df = pd.DataFrame(hfull_rows)
save_df(hfull_df, "test4_hfull_wrong_frequency_controls.csv")


# ============================================================
# TEST 5: FREQUENCY -> RESIDUE MATRIX + NORMALIZED SELECTIVITY
# ============================================================

print_section("TEST 5: FREQUENCY -> RESIDUE MATRIX")

CHANCE = {2: 0.5, 3: 1/3, 5: 0.2, 7: 1/7, 10: 0.1}
KS = (2, 3, 5, 7, 10)


def selectivity(r, ks=KS):
    return {k: (r[f"mod{k}"] - CHANCE[k]) / (1 - CHANCE[k]) for k in ks}


matrix_rows = []
print("RAW residue matrix")
print(f"{'ablate':9} {'exact':>7} " + " ".join(f"{'mod'+str(k):>7}" for k in KS))
print(f"{'none':9} {1.0:>7.3f} " + " ".join(f"{1.0:>7.3f}" for _ in KS))

for T in [2, 3, 5, 7, 10]:
    Q = {l: fit_Q(Xres[l], Sres, periods=[T], include_linear=False) for l in layers if l >= L0}
    r = eval_persistent(eval_examples, Q, label=f"matrix T{T}", ks=KS)
    row = {"ablate": f"T{T}", **r}
    row.update({f"sel_mod{k}": selectivity(r)[k] for k in KS})
    matrix_rows.append(row)
    print(f"T{T:<8} {r['exact']:>7.3f} " + " ".join(f"{r[f'mod{k}']:>7.3f}" for k in KS))

for T in [3, 7]:
    Qp = {l: orth_to(fit_Q(Xres[l], Sres, periods=[T], include_linear=False), Hfull[l])[0] for l in Hfull}
    r = eval_persistent(eval_examples, Qp, label=f"matrix T{T}_perp", ks=KS)
    row = {"ablate": f"T{T}_perp", **r}
    row.update({f"sel_mod{k}": selectivity(r)[k] for k in KS})
    matrix_rows.append(row)
    print(f"{'T'+str(T)+'_perp':9} {r['exact']:>7.3f} " + " ".join(f"{r[f'mod{k}']:>7.3f}" for k in KS))

matrix_df = pd.DataFrame(matrix_rows)
save_df(matrix_df, "test5_residue_matrix.csv")

print("\nNORMALIZED selectivity (acc-chance)/(1-chance)  [lower = more damaged]")
print(f"{'ablate':9} " + " ".join(f"{'mod'+str(k):>7}" for k in KS))
for _, row in matrix_df.iterrows():
    print(f"{row['ablate']:9} " + " ".join(f"{row[f'sel_mod{k}']:>7.3f}" for k in KS))


# ============================================================
# FINAL SUMMARY
# ============================================================

print_section("FINAL SUMMARY")
print("Run mode:", RUN_MODE)
print("Eval examples:", len(eval_examples))
print("Fit examples:", min(FIT_N, len(baseline_correct_examples)))
print("Output dir:", OUT_DIR if SAVE_RESULTS else None)
print("Done.")
print("\nInterpretation reminders:")
print("  - Strong causal claim: persistent low-period Fourier-span ablation damages addition/units digit.")
print("  - Strong residue claim: T5 ablation preserves mod2 but damages mod5/mod10.")
print("  - Safe phase claim must come from separate decode/phase-radius notebook, not from causal phase rotation.")
print("  - Do not claim T3/T7 are harmless; claim their raw damage is largely reduced after orthogonalization to empirical helix/magnitude span.")


In [ ]:
# ============================================================
# TEST 6: PIN / REPLACE STEERING — SUFFICIENCY TEST
#
# Positive causal test:
#   Instead of removing a Fourier subspace, replace/pin its current content
#   with the counterfactual Fourier content corresponding to s -> s + delta.
#
# Why pin instead of additive steering?
#   Additive steering accumulates across layers if applied persistently.
#   Pinning is bounded:
#
#       v' = v - Proj_Q(v) + target_Q_content
#
#   This mirrors persistent ablation:
#       ablation pins subspace content to zero
#       steering pins subspace content to counterfactual residue content
#
# Interpretation:
#   follow_k > stay_k means output residue follows the counterfactual
#   residue more than the original residue.
#
# Best success criterion:
#   T5-only, delta=1:
#       follow5 > stay5
#       stay2 remains high
#
# Requires existing globals:
#   model, tokenizer, BATCH_SIZE, RIDGE_ALPHA
#   Xres, Sres, layers, L0, eval_examples
#   make_fourier_basis, get_last_positions, pred_token_to_int
#   print_section, save_df
# ============================================================

def fit_QWB(X, sums, periods, include_linear=False):
    """
    Fit X ≈ Y @ W.T + bias, where Y is Fourier basis.
    Returns:
      Q    : orthonormal basis of learned Fourier span [D, K]
      W    : regression directions [D, K]
      bias : intercept [D]
    """
    Y = make_fourier_basis(sums, periods, include_linear)
    reg = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True).fit(Y, X)

    W = reg.coef_       # [D, K]
    b = reg.intercept_  # [D]

    Q, _ = np.linalg.qr(W)

    return (
        torch.tensor(Q, dtype=torch.float16, device=model.device),
        torch.tensor(W, dtype=torch.float16, device=model.device),
        torch.tensor(b, dtype=torch.float16, device=model.device),
    )


_st_lp = None
_st_bi = None
_st_Q = {}
_st_target = {}


def make_pin_hook(layer):
    def hook(module, inputs, output):
        global _st_lp, _st_bi, _st_Q, _st_target

        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        v = hs[_st_bi, _st_lp]  # [B, D]
        Q = _st_Q[layer]        # [D, K]

        # Remove current subspace content and replace it with target content.
        hs[_st_bi, _st_lp] = v - (v @ Q) @ Q.T + _st_target[layer]

        return hs if rest is None else (hs,) + rest

    return hook


@torch.no_grad()
def eval_pin_steer(
    rows,
    QWB_by_layer,
    delta,
    periods,
    alpha=1.0,
    ks=(2, 5, 10),
    label="",
):
    """
    Persistent pin steering.

    alpha:
      0.0 -> pin to ridge-predicted current Fourier content for s
      1.0 -> pin to ridge-predicted counterfactual content for s + delta
      between -> interpolate in Fourier-basis space
      >1 -> extrapolate beyond s+delta

    follow_k:
      pred % k == (true_sum + delta) % k

    stay_k:
      pred % k == true_sum % k
    """
    global _st_lp, _st_bi, _st_Q, _st_target

    handles = [
        model.gpt_neox.layers[l].register_forward_hook(make_pin_hook(l))
        for l in QWB_by_layer
    ]

    n = 0
    exact = 0
    parseable = 0
    follow = {k: 0 for k in ks}
    stay = {k: 0 for k in ks}
    shifts = []

    try:
        for start in tqdm(range(0, len(rows), BATCH_SIZE), desc=f"pin steer {label}"):
            batch = rows[start:start + BATCH_SIZE]

            s = np.array([x["sum"] for x in batch], dtype=np.float64)
            s_target = s + delta

            Ys = make_fourier_basis(s, periods, include_linear=False)
            Yt = make_fourier_basis(s_target, periods, include_linear=False)

            # Interpolate/extrapolate in Fourier basis.
            # alpha=0 -> current, alpha=1 -> target.
            Yint_np = Ys + alpha * (Yt - Ys)
            Yint = torch.tensor(Yint_np, dtype=torch.float16, device=model.device)

            enc = tokenizer(
                [x["prompt"] for x in batch],
                return_tensors="pt",
                padding=True,
                add_special_tokens=False,
            ).to(model.device)

            _st_lp = get_last_positions(enc["attention_mask"])
            _st_bi = torch.arange(len(batch), device=model.device)

            _st_Q = {}
            _st_target = {}

            for l, (Q, W, bias) in QWB_by_layer.items():
                # Ridge-predicted activation content for the counterfactual Fourier basis.
                # yhat = Yint @ W.T + bias  -> [B, D]
                yhat = Yint @ W.T + bias

                # Only pin content inside Q span.
                # This makes the intervention local to the learned Fourier subspace.
                target_q = (yhat @ Q) @ Q.T

                _st_Q[l] = Q
                _st_target[l] = target_q

            logits = model(**enc, use_cache=False).logits[_st_bi, _st_lp]
            pred_ids = logits.argmax(-1).detach().cpu().tolist()

            for pred_id, ex in zip(pred_ids, batch):
                n += 1
                exact += int(int(pred_id) == int(ex["target_token_id"]))

                pred_num = pred_token_to_int(pred_id)
                if pred_num is None:
                    continue

                parseable += 1
                shifts.append(pred_num - ex["sum"])

                for k in ks:
                    follow[k] += int(pred_num % k == int(ex["sum"] + delta) % k)
                    stay[k] += int(pred_num % k == ex["sum"] % k)

    finally:
        for h in handles:
            h.remove()

    n_safe = max(n, 1)

    out = {
        "label": label,
        "delta": delta,
        "alpha": alpha,
        "n": n,
        "exact": exact / n_safe,
        "parseable": parseable / n_safe,
        "median_shift": float(np.median(shifts)) if len(shifts) else None,
    }

    for k in ks:
        out[f"follow{k}"] = follow[k] / n_safe
        out[f"stay{k}"] = stay[k] / n_safe
        out[f"net{k}"] = (follow[k] - stay[k]) / n_safe

    return out


print_section("TEST 6: PIN / REPLACE STEERING — SUFFICIENCY")

# For smoke, use eval_examples[:1000].
# For paper run, use full eval_examples.
steer_eval = eval_examples

steer_rows = []

# ------------------------------------------------------------
# 6A. Whole low-period span [2,5,10]
#
# delta=1:
#   if successful, follow10 should rise and stay10 should fall.
#
# delta=10:
#   null-ish control for periods [2,5,10], because s and s+10
#   have identical residues under 2, 5, and 10.
#   It is not a pure no-op: it still pins to ridge-predicted current content.
#   Therefore alpha=0 and delta=10 are useful sanity checks for pinning damage.
# ------------------------------------------------------------

QWB_helix = {
    l: fit_QWB(Xres[l], Sres, periods=PERIODS_MAIN, include_linear=False)
    for l in layers
    if l >= L0
}

print("\n6A. Whole helix [2,5,10] pin steering")
print(
    f"{'delta':>5} {'alpha':>7} {'exact':>7} {'med_shift':>10} "
    f"{'follow10':>9} {'stay10':>7} {'net10':>7} "
    f"{'follow5':>8} {'stay5':>7} {'net5':>7} "
    f"{'follow2':>8} {'stay2':>7} {'net2':>7}"
)

# Include alpha=0 to measure pinning-to-current damage.
for delta in [1, 2, 5, 10]:
    for alpha in [0.0, 0.25, 0.5, 1.0, 1.5, 2.0]:
        r = eval_pin_steer(
            steer_eval,
            QWB_helix,
            delta=delta,
            periods=PERIODS_MAIN,
            alpha=alpha,
            ks=(2, 5, 10),
            label=f"helix_delta{delta}_alpha{alpha}",
        )
        r["mode"] = "helix_2_5_10"
        steer_rows.append(r)

        print(
            f"{delta:>5} {alpha:>7.2f} {r['exact']:>7.3f} {(r['median_shift'] or 0):>10.1f} "
            f"{r['follow10']:>9.3f} {r['stay10']:>7.3f} {r['net10']:>7.3f} "
            f"{r['follow5']:>8.3f} {r['stay5']:>7.3f} {r['net5']:>7.3f} "
            f"{r['follow2']:>8.3f} {r['stay2']:>7.3f} {r['net2']:>7.3f}"
        )


# ------------------------------------------------------------
# 6B. T5-only pin steering
#
# Best expected positive result:
#   delta=1, alpha≈1:
#     follow5 > stay5
#     stay2 remains high
#
# Since T5 does not specify parity, follow10 may not dominate;
# mod10 needs both mod2 and mod5.
# ------------------------------------------------------------

QWB_t5 = {
    l: fit_QWB(Xres[l], Sres, periods=[5], include_linear=False)
    for l in layers
    if l >= L0
}

print("\n6B. T5-only pin steering, delta=1")
print(
    f"{'alpha':>7} {'exact':>7} {'med_shift':>10} "
    f"{'follow5':>8} {'stay5':>7} {'net5':>7} "
    f"{'follow10':>9} {'stay10':>7} {'net10':>7} "
    f"{'follow2':>8} {'stay2':>7} {'net2':>7}"
)

for alpha in [0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 4.0]:
    r = eval_pin_steer(
        steer_eval,
        QWB_t5,
        delta=1,
        periods=[5],
        alpha=alpha,
        ks=(2, 5, 10),
        label=f"t5_delta1_alpha{alpha}",
    )
    r["mode"] = "t5_only"
    steer_rows.append(r)

    print(
        f"{alpha:>7.2f} {r['exact']:>7.3f} {(r['median_shift'] or 0):>10.1f} "
        f"{r['follow5']:>8.3f} {r['stay5']:>7.3f} {r['net5']:>7.3f} "
        f"{r['follow10']:>9.3f} {r['stay10']:>7.3f} {r['net10']:>7.3f} "
        f"{r['follow2']:>8.3f} {r['stay2']:>7.3f} {r['net2']:>7.3f}"
    )


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

steer_df = pd.DataFrame(steer_rows)
save_df(steer_df, "test6_pin_steering.csv")

print("\nInterpretation checklist:")
print("  Good helix steering: delta=1 has follow10 > stay10 and positive net10.")
print("  Good T5 steering: follow5 > stay5, with stay2 high.")
print("  alpha=0 checks pinning-to-current damage.")
print("  delta=10 checks null-ish behavior for periods [2,5,10].")

In [ ]:
# ============================================================
# FINAL AUDIT / DEBUG CELL
#
# Purpose:
#   Debug the parts of the experiment that actually matter,
#   without trying to manually inspect the whole notebook.
#
# Checks:
#   1. Metric correctness
#      - exact / mod-k definitions
#      - token exact vs integer exact agreement
#      - follow/stay arithmetic sanity
#
#   2. Intervention semantics
#      - hook fires at the intended layer
#      - only the intended last-token residual position is changed
#      - non-last positions are unchanged
#      - ablation equals v - Proj_Q(v)
#      - post-ablation vector has small remaining Q component
#
#   3. Subset framing
#      - eval_examples is actually a subset of baseline_correct_examples
#      - reminds you to report intervention results as baseline-correct eval
#
#   4. Control interpretation
#      - T2/T5/T10 damage is separated from matched-rank random controls
#      - T3/T7 raw damage is reduced after orthogonalization
#      - pin steering has delta=1 follow effect and delta=10 null-ish behavior
#
# Expected existing globals from the main notebook:
#   model, tokenizer, eval_examples, baseline_correct_examples,
#   BATCH_SIZE, get_last_positions, pred_token_to_int
#
# Optional globals used if available:
#   Qh, L0, periodwise_df, wrong_df, hfull_df, steer_df
# ============================================================

import numpy as np
import pandas as pd
import torch


# ============================================================
# REPORTING HELPERS
# ============================================================

AUDIT_RESULTS = []


def audit_status(name, status, message, details=None):
    """
    status should be one of:
      PASS, WARN, FAIL, INFO
    """
    AUDIT_RESULTS.append({
        "check": name,
        "status": status,
        "message": message,
        "details": details,
    })


def print_audit_results():
    print("\n" + "=" * 100)
    print("FINAL AUDIT SUMMARY")
    print("=" * 100)

    order = {"FAIL": 0, "WARN": 1, "PASS": 2, "INFO": 3}
    rows = sorted(AUDIT_RESULTS, key=lambda r: order.get(r["status"], 99))

    for r in rows:
        print(f"[{r['status']}] {r['check']}")
        print(f"  {r['message']}")
        if r["details"] is not None:
            print(f"  details: {r['details']}")
        print()

    print("=" * 100)
    n_fail = sum(r["status"] == "FAIL" for r in AUDIT_RESULTS)
    n_warn = sum(r["status"] == "WARN" for r in AUDIT_RESULTS)
    print(f"TOTAL: {len(AUDIT_RESULTS)} checks | FAIL={n_fail} WARN={n_warn}")
    print("=" * 100)


def require_globals(names, check_name):
    missing = [n for n in names if n not in globals()]
    if missing:
        audit_status(
            check_name,
            "FAIL",
            "Missing required globals.",
            {"missing": missing},
        )
        return False
    return True


# ============================================================
# 1. METRIC CORRECTNESS
# ============================================================

@torch.no_grad()
def audit_metric_correctness(n_examples=128, ks=(2, 3, 5, 7, 10)):
    check_name = "1. Metric correctness"

    needed = [
        "model", "tokenizer", "eval_examples", "BATCH_SIZE",
        "get_last_positions", "pred_token_to_int",
    ]
    if not require_globals(needed, check_name):
        return

    rows = eval_examples[:min(n_examples, len(eval_examples))]
    if len(rows) == 0:
        audit_status(check_name, "FAIL", "eval_examples is empty.")
        return

    exact_by_token = 0
    exact_by_int = 0
    parseable = 0
    mod_hits = {k: 0 for k in ks}
    bad_rows = []

    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]

        enc = tokenizer(
            [x["prompt"] for x in batch],
            return_tensors="pt",
            padding=True,
            add_special_tokens=False,
        ).to(model.device)

        lp = get_last_positions(enc["attention_mask"])
        bi = torch.arange(len(batch), device=model.device)

        logits = model(**enc, use_cache=False).logits[bi, lp]
        pred_ids = logits.argmax(-1).detach().cpu().tolist()

        for pred_id, ex in zip(pred_ids, batch):
            token_exact = int(pred_id) == int(ex["target_token_id"])
            exact_by_token += int(token_exact)

            pred_num = pred_token_to_int(pred_id)

            if pred_num is not None:
                parseable += 1
                int_exact = int(pred_num) == int(ex["sum"])
                exact_by_int += int(int_exact)

                for k in ks:
                    mod_hits[k] += int(pred_num % k == int(ex["sum"]) % k)

                # Since all answers in this dataset should be single-token answers,
                # token exact and integer exact should agree on baseline-correct rows.
                if token_exact != int_exact:
                    bad_rows.append({
                        "a": ex.get("a"),
                        "b": ex.get("b"),
                        "sum": ex.get("sum"),
                        "pred_id": int(pred_id),
                        "pred_num": pred_num,
                        "target_token_id": int(ex["target_token_id"]),
                        "token_exact": token_exact,
                        "int_exact": int_exact,
                    })
            else:
                if token_exact:
                    bad_rows.append({
                        "a": ex.get("a"),
                        "b": ex.get("b"),
                        "sum": ex.get("sum"),
                        "pred_id": int(pred_id),
                        "pred_num": None,
                        "target_token_id": int(ex["target_token_id"]),
                        "token_exact": token_exact,
                        "int_exact": None,
                    })

    n = len(rows)
    exact_token_rate = exact_by_token / n
    exact_int_rate = exact_by_int / n
    parse_rate = parseable / n
    mod_rates = {f"mod{k}": mod_hits[k] / n for k in ks}

    # Arithmetic-only sanity test for follow/stay definitions.
    # This does not use the model. It only checks the formulas.
    synthetic_ok = True
    for s in range(0, 50):
        pred_stay = s
        pred_follow = s + 1

        stay10 = pred_stay % 10 == s % 10
        follow10 = pred_follow % 10 == (s + 1) % 10
        stay5 = pred_stay % 5 == s % 5
        follow5 = pred_follow % 5 == (s + 1) % 5
        stay2 = pred_stay % 2 == s % 2
        follow2 = pred_follow % 2 == (s + 1) % 2

        if not all([stay10, follow10, stay5, follow5, stay2, follow2]):
            synthetic_ok = False
            break

    if len(bad_rows) == 0 and synthetic_ok:
        audit_status(
            check_name,
            "PASS",
            "Exact/mod/follow/stay arithmetic definitions look internally consistent on sampled rows.",
            {
                "n_checked": n,
                "exact_by_token": exact_token_rate,
                "exact_by_int": exact_int_rate,
                "parseable": parse_rate,
                **mod_rates,
            },
        )
    else:
        audit_status(
            check_name,
            "FAIL",
            "Metric mismatch found: token exact and integer exact disagree, or follow/stay arithmetic failed.",
            {
                "n_checked": n,
                "bad_examples_first_10": bad_rows[:10],
                "synthetic_follow_stay_ok": synthetic_ok,
            },
        )


# ============================================================
# 2. INTERVENTION SEMANTICS
# ============================================================

@torch.no_grad()
def audit_intervention_semantics(n_examples=8, layer=None, atol=5e-2, rel_atol=5e-4):
    """
    Verifies the hook locally:
      - only last positions are changed at the hooked layer output
      - non-last positions are unchanged inside the hook
      - last-position change equals -Proj_Q(v), within fp16 tolerance
      - post-ablation last-position vector has small remaining Q component

    Important:
      The model runs in fp16, so exact elementwise equality is too strict.
      A max absolute formula error around 1e-2 to 5e-2 can be harmless if:
        - non-last positions are unchanged
        - the relative error is tiny
        - the remaining Q component is tiny
    """
    check_name = "2. Intervention semantics"

    needed = ["model", "tokenizer", "eval_examples", "get_last_positions"]
    if not require_globals(needed, check_name):
        return

    if layer is None:
        layer = globals().get("L0", 17)

    if "Qh" in globals() and isinstance(Qh, dict) and layer in Qh:
        Q = Qh[layer]
        q_source = f"Qh[{layer}]"
    else:
        # Fallback random Q is enough to test hook mechanics,
        # but not the scientific meaning of the actual ablation.
        D = model.config.hidden_size
        K = 6
        A = torch.randn(D, K, device=model.device, dtype=torch.float32)
        Qtmp, _ = torch.linalg.qr(A)
        Q = Qtmp.to(dtype=torch.float16)
        q_source = "fallback random Q"

    rows = eval_examples[:min(n_examples, len(eval_examples))]
    if len(rows) == 0:
        audit_status(check_name, "FAIL", "eval_examples is empty.")
        return

    enc = tokenizer(
        [x["prompt"] for x in rows],
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(model.device)

    lp = get_last_positions(enc["attention_mask"])
    bi = torch.arange(len(rows), device=model.device)

    record = {}

    def audit_hook(module, inputs, output):
        if isinstance(output, tuple):
            hs = output[0].clone()
            rest = tuple(output[1:])
        else:
            hs = output.clone()
            rest = None

        before = hs.detach().clone()

        v = hs[bi, lp]
        projected = (v @ Q) @ Q.T
        hs[bi, lp] = v - projected

        after = hs.detach().clone()

        record["before"] = before.float().cpu()
        record["after"] = after.float().cpu()
        record["lp"] = lp.detach().cpu()
        record["projected"] = projected.detach().float().cpu()
        record["Q"] = Q.detach().float().cpu()

        return hs if rest is None else (hs,) + rest

    handle = model.gpt_neox.layers[layer].register_forward_hook(audit_hook)
    try:
        _ = model(**enc, use_cache=False)
    finally:
        handle.remove()

    if "before" not in record:
        audit_status(
            check_name,
            "FAIL",
            "Hook did not fire.",
            {"layer": layer},
        )
        return

    before = record["before"]
    after = record["after"]
    lp_cpu = record["lp"]
    projected = record["projected"]
    Q_cpu = record["Q"]

    B, T, D = before.shape

    nonlast_max_abs_change = 0.0
    last_formula_max_abs_err = 0.0
    last_formula_max_rel_err = 0.0
    post_q_rel_norms = []
    changed_token_norms = []

    for i in range(B):
        mask = torch.ones(T, dtype=torch.bool)
        mask[int(lp_cpu[i])] = False

        nonlast_delta = (after[i, mask] - before[i, mask]).abs().max().item()
        nonlast_max_abs_change = max(nonlast_max_abs_change, nonlast_delta)

        expected_last = before[i, int(lp_cpu[i])] - projected[i]
        err_vec = after[i, int(lp_cpu[i])] - expected_last
        formula_abs_err = err_vec.abs().max().item()
        last_formula_max_abs_err = max(last_formula_max_abs_err, formula_abs_err)

        expected_norm = torch.linalg.norm(expected_last).item() + 1e-12
        formula_rel_err = torch.linalg.norm(err_vec).item() / expected_norm
        last_formula_max_rel_err = max(last_formula_max_rel_err, formula_rel_err)

        delta_norm = torch.linalg.norm(after[i, int(lp_cpu[i])] - before[i, int(lp_cpu[i])]).item()
        changed_token_norms.append(delta_norm)

        post = after[i, int(lp_cpu[i])]
        q_component_norm = torch.linalg.norm(post @ Q_cpu).item()
        post_norm = torch.linalg.norm(post).item() + 1e-12
        post_q_rel_norms.append(q_component_norm / post_norm)

    max_post_q_rel_norm = max(post_q_rel_norms)
    median_changed_token_norm = float(np.median(changed_token_norms))

    ok_nonlast = nonlast_max_abs_change <= 1e-6
    ok_formula = (last_formula_max_abs_err <= atol) or (last_formula_max_rel_err <= rel_atol)
    ok_q = max_post_q_rel_norm < 5e-3

    if ok_nonlast and ok_formula and ok_q:
        status = "PASS"
        msg = "Hook changes only intended last positions and performs the expected projection removal within fp16 tolerance."
    elif ok_nonlast and ok_q:
        status = "WARN"
        msg = "Hook target/position semantics look correct, but the formula equality tolerance is loose. This is often fp16 rounding; inspect details."
    else:
        status = "FAIL"
        msg = "Hook semantics check failed. Inspect non-last changes, formula error, or remaining Q component."

    audit_status(
        check_name,
        status,
        msg,
        {
            "layer": layer,
            "Q_source": q_source,
            "batch": B,
            "sequence_length": T,
            "nonlast_max_abs_change": nonlast_max_abs_change,
            "last_formula_max_abs_err": last_formula_max_abs_err,
            "last_formula_max_rel_err": last_formula_max_rel_err,
            "max_post_q_relative_norm": max_post_q_rel_norm,
            "median_changed_token_norm": median_changed_token_norm,
            "abs_atol": atol,
            "rel_atol": rel_atol,
            "interpretation": "If nonlast=0 and max_post_q_relative_norm is tiny, a small fp16 absolute formula error is usually not a real intervention bug.",
        },
    )


# ============================================================
# 3. SUBSET FRAMING
# ============================================================

def audit_subset_framing():
    check_name = "3. Subset framing"

    needed = ["eval_examples", "baseline_correct_examples"]
    if not require_globals(needed, check_name):
        return

    n_eval = len(eval_examples)
    n_base_correct = len(baseline_correct_examples)

    if n_eval == 0:
        audit_status(check_name, "FAIL", "eval_examples is empty.")
        return

    eval_ids = {(x.get("a"), x.get("b"), x.get("sum")) for x in eval_examples}
    base_ids = {(x.get("a"), x.get("b"), x.get("sum")) for x in baseline_correct_examples}
    is_subset = eval_ids.issubset(base_ids)

    correct_flags = [x.get("correct", None) for x in eval_examples]
    has_correct_flags = all(v is not None for v in correct_flags)
    all_marked_correct = all(bool(v) for v in correct_flags) if has_correct_flags else None

    if is_subset and (all_marked_correct is True or not has_correct_flags):
        audit_status(
            check_name,
            "PASS",
            "eval_examples is a subset of baseline_correct_examples. Report intervention accuracies as baseline-correct eval accuracy.",
            {
                "n_eval_examples": n_eval,
                "n_baseline_correct_examples": n_base_correct,
                "has_correct_flags": has_correct_flags,
                "all_marked_correct": all_marked_correct,
                "recommended_phrase": "All intervention results are evaluated on baseline-correct examples.",
            },
        )
    else:
        audit_status(
            check_name,
            "FAIL",
            "eval_examples is not cleanly verified as baseline-correct.",
            {
                "n_eval_examples": n_eval,
                "n_baseline_correct_examples": n_base_correct,
                "is_subset": is_subset,
                "has_correct_flags": has_correct_flags,
                "all_marked_correct": all_marked_correct,
            },
        )


# ============================================================
# 4. CONTROL INTERPRETATION
# ============================================================

def audit_control_interpretation():
    check_name = "4. Control interpretation"

    found_any = False

    # --------------------------------------------------------
    # 4A. Random rank-6 should be much less damaging than T2T5T10.
    # --------------------------------------------------------
    if "periodwise_df" in globals():
        found_any = True
        df = periodwise_df.copy()

        try:
            t_main = df[df["subspace"] == "T2T5T10"].iloc[0]
            rand = df[df["subspace"] == "RANDOM_rank6"]

            if len(rand) == 0:
                audit_status(
                    check_name,
                    "WARN",
                    "periodwise_df exists, but RANDOM_rank6 rows are missing.",
                )
            else:
                rand_exact_mean = float(rand["exact"].mean())
                rand_m10_mean = float(rand["mod10"].mean())
                main_exact = float(t_main["exact"])
                main_m10 = float(t_main["mod10"])

                if rand_exact_mean - main_exact > 0.5 and rand_m10_mean - main_m10 > 0.5:
                    audit_status(
                        check_name,
                        "PASS",
                        "Matched-rank random controls are much less damaging than T2/T5/T10 ablation.",
                        {
                            "T2T5T10_exact": main_exact,
                            "random_rank6_exact_mean": rand_exact_mean,
                            "T2T5T10_mod10": main_m10,
                            "random_rank6_mod10_mean": rand_m10_mean,
                            "safe_interpretation": "Rank-matched random removals are substantially less damaging.",
                            "do_not_say": "Random controls prove the span is unique or complete.",
                        },
                    )
                else:
                    audit_status(
                        check_name,
                        "WARN",
                        "Random controls are not clearly separated from T2/T5/T10. Be conservative.",
                        {
                            "T2T5T10_exact": main_exact,
                            "random_rank6_exact_mean": rand_exact_mean,
                            "T2T5T10_mod10": main_m10,
                            "random_rank6_mod10_mean": rand_m10_mean,
                        },
                    )
        except Exception as e:
            audit_status(
                check_name,
                "WARN",
                "Could not parse periodwise_df for random-control audit.",
                {"error": repr(e)},
            )

    # --------------------------------------------------------
    # 4B. Wrong-frequency controls should improve after orthogonalization.
    # --------------------------------------------------------
    if "wrong_df" in globals():
        found_any = True
        df = wrong_df.copy()

        for T in [3, 7]:
            try:
                raw = df[(df["freq"] == T) & (df["condition"] == "raw")].iloc[0]
                orth = df[(df["freq"] == T) & (df["condition"].astype(str).str.contains("orth"))].iloc[0]

                raw_exact = float(raw["exact"])
                orth_exact = float(orth["exact"])
                raw_m10 = float(raw["mod10"])
                orth_m10 = float(orth["mod10"])

                if orth_exact > raw_exact and orth_m10 > raw_m10:
                    status = "PASS"
                    msg = f"T{T} damage is reduced after orthogonalization."
                else:
                    status = "WARN"
                    msg = f"T{T} orthogonalization does not clearly reduce damage."

                audit_status(
                    check_name,
                    status,
                    msg,
                    {
                        "freq": T,
                        "raw_exact": raw_exact,
                        "orth_exact": orth_exact,
                        "raw_mod10": raw_m10,
                        "orth_mod10": orth_m10,
                        "safe_interpretation": "Raw wrong-frequency damage is partly explained by overlap/leakage.",
                        "do_not_say": "T3/T7 are harmless or irrelevant.",
                    },
                )
            except Exception as e:
                audit_status(
                    check_name,
                    "WARN",
                    f"Could not parse wrong_df for T{T}.",
                    {"error": repr(e)},
                )

    # --------------------------------------------------------
    # 4C. Hfull wrong-frequency controls, if available.
    # --------------------------------------------------------
    if "hfull_df" in globals():
        found_any = True
        df = hfull_df.copy()

        for T in [3, 7]:
            try:
                raw = df[(df["freq"] == T) & (df["condition"] == "raw")].iloc[0]
                orth = df[(df["freq"] == T) & (df["condition"].astype(str).str.contains("orth"))].iloc[0]

                raw_exact = float(raw["exact"])
                orth_exact = float(orth["exact"])
                raw_m10 = float(raw["mod10"])
                orth_m10 = float(orth["mod10"])

                if orth_exact > raw_exact and orth_m10 > raw_m10:
                    status = "PASS"
                    msg = f"T{T} Hfull-orthogonalized control reduces raw damage."
                else:
                    status = "WARN"
                    msg = f"T{T} Hfull orthogonalization does not clearly reduce damage."

                audit_status(
                    check_name,
                    status,
                    msg,
                    {
                        "freq": T,
                        "raw_exact": raw_exact,
                        "orth_exact": orth_exact,
                        "raw_mod10": raw_m10,
                        "orth_mod10": orth_m10,
                    },
                )
            except Exception as e:
                audit_status(
                    check_name,
                    "WARN",
                    f"Could not parse hfull_df for T{T}.",
                    {"error": repr(e)},
                )

    # --------------------------------------------------------
    # 4D. Steering sanity, if available.
    # --------------------------------------------------------
    if "steer_df" in globals():
        found_any = True
        df = steer_df.copy()

        try:
            helix_d1 = df[
                (df["mode"] == "helix_2_5_10") &
                (df["delta"] == 1) &
                (df["alpha"] == 1.0)
            ].iloc[0]

            helix_d10 = df[
                (df["mode"] == "helix_2_5_10") &
                (df["delta"] == 10) &
                (df["alpha"] == 1.0)
            ].iloc[0]

            d1_ok = float(helix_d1["follow10"]) > float(helix_d1["stay10"])
            d10_nullish = abs(float(helix_d10["follow10"]) - float(helix_d10["stay10"])) < 1e-6

            if d1_ok and d10_nullish:
                status = "PASS"
                msg = "Pin steering has the expected delta=1 follow effect and delta=10 null-ish behavior."
            else:
                status = "WARN"
                msg = "Pin steering sanity pattern is not clean; interpret sufficiency-style claims conservatively."

            audit_status(
                check_name,
                status,
                msg,
                {
                    "delta1_alpha1_follow10": float(helix_d1["follow10"]),
                    "delta1_alpha1_stay10": float(helix_d1["stay10"]),
                    "delta10_alpha1_follow10": float(helix_d10["follow10"]),
                    "delta10_alpha1_stay10": float(helix_d10["stay10"]),
                    "safe_interpretation": "Pin/replacement steering provides sufficiency-style evidence.",
                    "do_not_say": "This proves the exact model algorithm.",
                },
            )
        except Exception as e:
            audit_status(
                check_name,
                "WARN",
                "Could not parse steer_df for steering sanity audit.",
                {"error": repr(e)},
            )

    if not found_any:
        audit_status(
            check_name,
            "WARN",
            "No result dataframes found. Expected optional globals: periodwise_df, wrong_df, hfull_df, steer_df.",
        )


# ============================================================
# RUN ALL AUDITS
# ============================================================

AUDIT_RESULTS.clear()

audit_metric_correctness(n_examples=128)
audit_intervention_semantics(n_examples=8, layer=globals().get("L0", 17))
audit_subset_framing()
audit_control_interpretation()

print_audit_results()
